In [1]:
print('hi')

hi


In [2]:
import requests
import pandas as pd

url = "https://www.gdacs.org/gdacsapi/api/events/geteventlist/SEARCH"
params = {
    "eventlist": "EQ",          # EQ=earthquake, TC=cyclone, FL=flood
    "fromdate": "2025-10-01",
    "todate": "2026-04-03",
    "alertlevel": "red;orange"
}

r = requests.get(url, params=params, timeout=30)
r.raise_for_status()

data = r.json()
type(data), list(data)[:5]

(dict, ['type', 'features', 'bbox'])

In [3]:
data

{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'bbox': [89.1394, 22.4513, 89.1394, 22.4513],
   'geometry': {'type': 'Point', 'coordinates': [89.1394, 22.4513]},
   'properties': {'eventtype': 'EQ',
    'eventid': 1526507,
    'episodeid': 1690089,
    'eventname': '',
    'glide': 'EQ-2026-000025-BGD',
    'name': 'Earthquake in Bangladesh',
    'description': 'Earthquake in Bangladesh',
    'htmldescription': 'Orange M 5.3 Earthquake in Bangladesh at: 27 Feb 2026 07:52:24.',
    'icon': 'https://www.gdacs.org/images/gdacs_icons/maps/Orange/EQ.png',
    'iconoverall': 'https://www.gdacs.org/images/gdacs_icons/maps/Orange/EQ.png',
    'url': {'geometry': 'https://www.gdacs.org/gdacsapi/api/polygons/getgeometry?eventtype=EQ&eventid=1526507&episodeid=1690089',
     'report': 'https://www.gdacs.org/report.aspx?eventid=1526507&episodeid=1690089&eventtype=EQ',
     'details': 'https://www.gdacs.org/gdacsapi/api/events/geteventdata?eventtype=EQ&eventid=1526507'},
    'ale

In [4]:
df = pd.json_normalize(data)
df.head()

,type,features,bbox
0,FeatureCollection,"[{'type': 'Feature', 'bbox': [89.1394, 22.4513...",None


In [5]:
data.keys()

dict_keys(['type', 'features', 'bbox'])

In [8]:
import requests
import json
import pandas as pd
from datetime import datetime

url = "https://www.gdacs.org/gdacsapi/api/events/geteventlist/SEARCH"
params = {
    "eventlist": "FL,TC,EQ",
    "fromdate": "2025-01-01",
    "todate": datetime.now().strftime("%Y-%m-%d"),
    "alertlevel": "orange;red"
}

r = requests.get(url, params=params)
data = r.json()

# Inspect first
print("Keys:", data.keys())
print("Sample feature:", json.dumps(data["features"][0], indent=2) if "features" in data else "No features")

# Normalize the features list
df = pd.json_normalize(data["features"])

# Filter for India/Chennai-relevant (safer column names)
local_cols = ['properties.country', 'properties.description', 'properties.htmldescription']
local = df[df['properties.country'].str.contains('India', na=False) | 
           df['properties.htmldescription'].str.contains('Tamil Nadu|Chennai', na=False, case=False)]

local.to_csv("chennai_gdacs.csv", index=False)
local[['properties.eventname', 'properties.country', 'properties.alertlevel', 'properties.fromdate']].head()

Keys: dict_keys(['type', 'features', 'bbox'])
Sample feature: {
  "type": "Feature",
  "bbox": [
    37.2620697,
    6.1701144,
    37.2620697,
    6.1701144
  ],
  "geometry": {
    "type": "Point",
    "coordinates": [
      37.2620697,
      6.1701144
    ]
  },
  "properties": {
    "eventtype": "FL",
    "eventid": 1103798,
    "episodeid": 1,
    "eventname": "",
    "glide": "FL-2026-000033-ETH",
    "name": "Flood in Ethiopia",
    "description": "Flood in Ethiopia",
    "htmldescription": "Orange Flood in Ethiopia from: 09 Mar 2026 01 to: 11 Mar 2026 01.",
    "icon": "https://www.gdacs.org/images/gdacs_icons/maps/Orange/FL.png",
    "iconoverall": "https://www.gdacs.org/images/gdacs_icons/maps/Orange/FL.png",
    "url": {
      "geometry": "https://www.gdacs.org/gdacsapi/api/polygons/getgeometry?eventtype=FL&eventid=1103798&episodeid=1",
      "report": "https://www.gdacs.org/report.aspx?eventid=1103798&episodeid=1&eventtype=FL",
      "details": "https://www.gdacs.org/gdacsa

,properties.eventname,properties.country,properties.alertlevel,properties.fromdate
12,DITWAH-25,"Sri Lanka, India",Orange,2025-11-27T00:00:00
19,MONTHA-25,India,Orange,2025-10-26T12:00:00
27,,India,Orange,2025-08-16T01:00:00
30,ONE-25,India,Orange,2025-10-01T06:00:00
41,,India,Orange,2025-09-14T11:11:51


In [10]:
import requests
import json
from datetime import datetime, timedelta

BASE_URL = "https://www.gdacs.org/gdacsapi/api/events/geteventlist/SEARCH"

def fetch_gdacs(params):
    r = requests.get(BASE_URL, params=params, timeout=30)
    if r.status_code == 204:
        return {"type": "FeatureCollection", "features": []}
    r.raise_for_status()
    return r.json()

def clean_event(feature):
    props = feature.get("properties", {})
    coords = feature.get("geometry", {}).get("coordinates", [None, None])

    return {
        "eventType": props.get("eventtype"),
        "eventId": props.get("eventid"),
        "episodeId": props.get("episodeid"),
        "name": props.get("name"),
        "description": props.get("description"),
        "alertLevel": props.get("alertlevel"),
        "country": props.get("country"),
        "fromDate": props.get("fromdate"),
        "toDate": props.get("todate"),
        "isCurrent": props.get("iscurrent"),
        "source": props.get("source"),
        "latitude": coords[1] if len(coords) > 1 else None,
        "longitude": coords[0] if len(coords) > 0 else None,
        "reportUrl": props.get("url", {}).get("report"),
        "detailsUrl": props.get("url", {}).get("details"),
    }

today = datetime.utcnow().date()
from_date = today - timedelta(days=30)

# 1) Try Chennai-area search first
local_params = {
    "eventlist": "FL;TC",
    "fromdate": str(from_date),
    "todate": str(today),
    "alertlevel": "green;orange;red",
    "country": "India",
    "latmin": 12.5,
    "latmax": 13.5,
    "lonmin": 79.5,
    "lonmax": 81.0,
    "pagesize": 10,
    "pagenumber": 1,
}

data = fetch_gdacs(local_params)
features = data.get("features", [])

# 2) Fallback to latest India-wide flood/storm if Chennai area has nothing
if not features:
    india_params = {
        "eventlist": "FL;TC",
        "fromdate": str(from_date),
        "todate": str(today),
        "alertlevel": "green;orange;red",
        "country": "India",
        "pagesize": 10,
        "pagenumber": 1,
    }
    data = fetch_gdacs(india_params)
    features = data.get("features", [])

# 3) Return latest one event as JSON
if features:
    latest_event = clean_event(features[0])
    print(json.dumps(latest_event, indent=2))
else:
    print(json.dumps({"message": "No recent flood or tropical cyclone found."}, indent=2))

{
  "message": "No recent flood or tropical cyclone found."
}


In [12]:
import requests
import json

BASE_URL = "https://www.gdacs.org/gdacsapi/api/events/geteventlist/SEARCH"

def fetch_events(event_type):
    params = {
        "eventlist": event_type,          # FL or TC
        "fromdate": "2025-01-01",
        "todate": "2026-04-03",
        "country": "India",
        "pagesize": 10,
        "pagenumber": 1
    }
    try:
        r = requests.get(BASE_URL, params=params, timeout=20)
        print(f"Fetching {event_type}: status {r.status_code}")
        if r.status_code == 204:
            print(f"No content for {event_type}")
            return []
        r.raise_for_status()
        data = r.json()
        features = data.get("features", [])
        print(f"Found {len(features)} {event_type} events")
        return features
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {event_type}: {e}")
        return []

def clean_event(feature):
    props = feature.get("properties", {})
    coords = feature.get("geometry", {}).get("coordinates", [None, None])

    return {
        "eventType": props.get("eventtype"),
        "eventId": props.get("eventid"),
        "episodeId": props.get("episodeid"),
        "name": props.get("name"),
        "description": props.get("description"),
        "alertLevel": props.get("alertlevel"),
        "country": props.get("country"),
        "fromDate": props.get("fromdate"),
        "toDate": props.get("todate"),
        "isCurrent": props.get("iscurrent"),
        "source": props.get("source"),
        "latitude": coords[1] if len(coords) > 1 else None,
        "longitude": coords[0] if len(coords) > 0 else None,
        "reportUrl": props.get("url", {}).get("report"),
        "detailsUrl": props.get("url", {}).get("details")
    }

# Fetch floods and cyclones separately
floods = fetch_events("FL")
cyclones = fetch_events("TC")

all_events = floods + cyclones
print(f"Total events: {len(all_events)}")

if not all_events:
    print(json.dumps({"message": "No recent flood or tropical cyclone found."}, indent=2))
else:
    # Sort by toDate descending
    all_events.sort(
        key=lambda f: f.get("properties", {}).get("todate", ""),
        reverse=True
    )
    latest_event = clean_event(all_events[0])
    print(json.dumps(latest_event, indent=2))

Fetching FL: status 200
Found 1 FL events
Fetching TC: status 200
Found 3 TC events
Total events: 4
{
  "eventType": "TC",
  "eventId": 1001238,
  "episodeId": 12,
  "name": "Tropical Cyclone DITWAH-25",
  "description": "Tropical Cyclone DITWAH-25",
  "alertLevel": "Orange",
  "country": "Sri Lanka, India",
  "fromDate": "2025-11-27T00:00:00",
  "toDate": "2025-11-29T18:00:00",
  "isCurrent": "false",
  "source": "JTWC",
  "latitude": 10.5,
  "longitude": 80.8,
  "reportUrl": "https://www.gdacs.org/report.aspx?eventid=1001238&episodeid=12&eventtype=TC",
  "detailsUrl": "https://www.gdacs.org/gdacsapi/api/events/geteventdata?eventtype=TC&eventid=1001238"
}
